# Lab21 — Lượt chính thức có sao lưu Google Drive

1. Chọn **Runtime → Change runtime type → T4 GPU**.
2. Chạy các ô từ trên xuống. Ô 1 cần bạn cho phép Colab kết nối Drive.
3. Giữ `EVAL_LIMIT=0`: đánh giá đầy đủ 50 ticket và 15 câu hỏi.
4. Mỗi notebook hoàn tất sẽ lưu `results/`, adapter, split dữ liệu, code và log vào **My Drive/Lab21/<lượt chạy>**.
5. Khi thấy `BACKUP SAVED`, bước đó đã được lưu ngoài VM. Lỗi report chưa điền xử lý sau.

Bản này giữ quy trình NB1–NB5. NB6 được sửa để giải phóng model cũ và bỏ xuất trọng số merged khoảng 9 GB; vẫn giữ kiểm tra điểm merge và hot-swap. B4 đã có script hỗ trợ nhưng chưa tự chạy.

Không dùng nút xóa runtime khi đang chạy. Nếu VM mất, các bước đã báo `BACKUP SAVED` vẫn nằm trên Drive.


In [1]:
# @title 1. Kết nối Google Drive
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
# @title 2. Setup + Smoke — chạy một lần cho lượt mới
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import os, subprocess, sys, uuid

assert Path('/content/drive/MyDrive').is_dir(), 'Chạy ô kết nối Drive trước.'
RUN_ID = datetime.now(ZoneInfo('Asia/Ho_Chi_Minh')).strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:6]
ROOT = Path('/content') / ('lab21_' + RUN_ID)
BACKUP = Path('/content/drive/MyDrive/Lab21') / RUN_ID
REPO = 'https://github.com/TNTD-dev/Day21-Track3-TranNguyenTienDuc-2A202602871-Finetuning-Lab.git'
subprocess.run(['git', 'clone', '-q', REPO, str(ROOT)], check=True)
os.chdir(ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
import torch
assert torch.cuda.is_available(), 'Chọn Runtime > Change runtime type > T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0))
print('Lượt mới:', ROOT)
print('Sao lưu Drive:', BACKUP)

(ROOT / "scripts/colab_run_persistent.py").write_text('"""Run core stages and save completed work outside the temporary Colab VM.\n\nUsage: python scripts/colab_run_persistent.py --backup-dir /content/drive/MyDrive/... nb1 nb2 nb3 nb4 nb5\nThe destination must be unique per experiment. Only successfully completed stages\nupdate the backup; a failed stage keeps the preceding completed backup intact.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport sys\n\n\nSTAGES = {f"nb{i}": path for i, path in enumerate([\n    "01_data_and_mask.py", "02_baselines.py", "03_train_correct.py",\n    "04_misconfig_autopsy.py", "05_evaluate_and_verdict.py",\n    "06_merge_and_serve.py"], start=1)}\n\n\ndef save_completed(root: Path, destination: Path):\n    for name in ("results", "adapters", "submission", "notebooks", "src", "scripts", "data"):\n        source = root / name\n        if source.is_dir():\n            shutil.copytree(source, destination / name, dirs_exist_ok=True,\n                            ignore=shutil.ignore_patterns("merged", "checkpoint-*", "__pycache__"))\n    for name in ("requirements.txt", "requirements-cpu.txt", "pyproject.toml", "Makefile", ".env.example"):\n        if (root / name).is_file():\n            shutil.copy2(root / name, destination / name)\n\n\ndef run(root: Path, destination: Path, stages: list[str]):\n    if any(stage not in STAGES for stage in stages):\n        raise ValueError("Unknown notebook stage.")\n    destination = destination.resolve()\n    root = root.resolve()\n    if destination == root or root in destination.parents:\n        raise ValueError("Backup must be outside the repo, preferably on mounted Drive.")\n    destination.mkdir(parents=True, exist_ok=True)\n    progress = destination / "progress.json"\n    manifest = json.loads(progress.read_text()) if progress.exists() else {"completed": []}\n    for stage in stages:\n        print(f"\\n{stage.upper()} — running; log: {destination / (stage + \'.log\')}", flush=True)\n        with (destination / f"{stage}.log").open("a", encoding="utf-8") as log:\n            child = subprocess.Popen(\n                [sys.executable, "-u", str(root / "notebooks" / STAGES[stage])],\n                cwd=root, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,\n                text=True, errors="replace")\n            for line in child.stdout:\n                print(line, end="", flush=True)\n                log.write(line)\n                log.flush()\n            code = child.wait()\n        if code:\n            manifest["failed_stage"] = stage\n            progress.write_text(json.dumps(manifest, indent=2))\n            print(f"{stage} failed. The last completed backup is preserved at {destination}", flush=True)\n            return code\n        # Save each completed notebook before starting the next GPU operation.\n        save_completed(root, destination)\n        if stage not in manifest["completed"]:\n            manifest["completed"].append(stage)\n        manifest.pop("failed_stage", None)\n        progress.write_text(json.dumps(manifest, indent=2))\n        print(f"BACKUP SAVED after {stage}: {destination}", flush=True)\n    return 0\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--backup-dir", required=True, type=Path)\n    parser.add_argument("stages", nargs="*", default=list(STAGES)[:5])\n    args = parser.parse_args()\n    root = Path(__file__).resolve().parents[1]\n    return run(root, args.backup_dir, args.stages or list(STAGES)[:5])\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', encoding="utf-8")
(ROOT / "scripts/bonus_rank_sweep.py").write_text('"""B4: train ranks 8/64 and reuse the existing rank-16 core run.\n\nRun from the lab repo on Colab: python /content/bonus_rank_sweep.py\nOnly bonus adapters/results are written. Each GPU operation uses a fresh process.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport runpy\nimport subprocess\nimport sys\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding="utf-8"))\n\n\ndef digest(path):\n    h = hashlib.sha256()\n    with Path(path).open("rb") as f:\n        for chunk in iter(lambda: f.read(1024 * 1024), b""):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef context(root):\n    required = ["results/baselines_frozen.json", "results/runs.csv", "results/verdict.json",\n                "adapters/correct/adapter_config.json", "adapters/correct/adapter_model.safetensors",\n                "data/split/train.jsonl", "data/split/val.jsonl"]\n    missing = [name for name in required if not (root / name).is_file()]\n    if missing:\n        raise RuntimeError(\n            "Saved core files are missing:\\n  " + "\\n  ".join(missing)\n            + "\\nRestore the full core ZIP in this repo before running B4. "\n            "A downloaded .ipynb contains logs, but not results/ or adapter weights. "\n            "No bonus training has started."\n        )\n    frozen = read_json(root / "results/baselines_frozen.json")\n    if frozen.get("smoke_mode"):\n        raise RuntimeError("Finish the full core evaluation before B4.")\n    with (root / "results/runs.csv").open(encoding="utf-8") as f:\n        rows = list(csv.DictReader(f))\n    core = next(r for r in reversed(rows) if r["run"] == "correct")\n    cfg = read_json(root / "adapters/correct/adapter_config.json")\n    if (int(core["r"]) != 16 or cfg["r"] != 16 or cfg["lora_alpha"] != 32\n            or core["placement"] != "text-linear"\n            or core["load_in_4bit"].lower() != "false"):\n        raise RuntimeError("B4 requires the existing 16-bit text-linear rank-16 run.")\n    if core["model"] != frozen["model"]:\n        raise RuntimeError("Core adapter and baselines use different models.")\n    paths = ["data/split/train.jsonl", "data/split/val.jsonl",\n             "data/eval_target.jsonl", "data/eval_regression.jsonl",\n             "results/baselines_frozen.json", "results/verdict.json", "results/runs.csv",\n             "adapters/correct/adapter_config.json", "adapters/correct/adapter_model.safetensors",\n             "notebooks/03_train_correct.py", "notebooks/05_evaluate_and_verdict.py"]\n    paths += [str(p.relative_to(root)) for p in sorted((root / "src/labkit").glob("*.py"))]\n    hashes = {name: digest(root / name) for name in paths}\n    for group, field in [("target", "n_target"), ("regression", "n_regression")]:\n        count = len((root / f"data/eval_{group}.jsonl").read_text().splitlines())\n        if count != frozen[field]:\n            raise RuntimeError(f"The {group} set differs from the frozen full evaluation.")\n    payload = {"core": core, "hashes": hashes}\n    payload["fingerprint"] = hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()\n    return payload\n\n\ndef require_same(saved, current):\n    if saved.get("fingerprint") != current["fingerprint"]:\n        raise RuntimeError("Existing bonus evidence uses a different core/config/dataset. "\n                           "Keep it separate; do not mix experiments.")\n\n\ndef configure(root, ctx):\n    sys.path.insert(0, str(root / "src"))\n    core = ctx["core"]\n    os.environ.update(COMPUTE_TIER=core["tier"], BASE_MODEL=core["model"],\n                      MASK_MODE=core.get("mask_mode") or "assistant-only", EVAL_LIMIT="0")\n    from labkit.config import get_tier\n    tier = get_tier()\n    n = len((root / "data/split/train.jsonl").read_text().splitlines())\n    steps = int(core["max_steps"])\n    per_epoch = math.ceil(n / tier.effective_batch)\n    # Reuse NB3\'s epoch recipe only when it reproduces the recorded step budget exactly.\n    epochs = steps / per_epoch\n    if not epochs.is_integer():\n        raise RuntimeError("The existing step budget is not an integer number of epochs.")\n    os.environ["EPOCHS"] = str(int(epochs))\n    return tier\n\n\ndef train_rank(root, ctx, rank):\n    from dataclasses import replace\n    configure(root, ctx)\n    from labkit import report\n    from labkit.config import SPECS\n    if float(ctx["core"]["learning_rate"]) != SPECS["correct"].lr:\n        raise RuntimeError("The core LR differs from the NB3 recipe.")\n    key = f"bonus_rank_{rank}"\n    SPECS["correct"] = replace(SPECS["correct"], key=key, r=rank, alpha=2 * rank,\n                               label=f"text-linear · r={rank} · controlled rank sweep")\n    original_append = report.append_row\n\n    def append_bonus(row, filename="runs.csv", results_dir=None):\n        return original_append(row, filename="bonus_rank_runs.csv", results_dir=results_dir)\n\n    report.append_row = append_bonus\n    state = runpy.run_path(str(root / "notebooks/03_train_correct.py"), run_name="__main__")\n    if state["STEPS"] != int(ctx["core"]["max_steps"]):\n        raise RuntimeError("Rank run did not use the core step budget.")\n    report.write_json(ctx, "bonus_context.json", results_dir=root / "adapters" / key)\n\n\ndef score_rank(root, ctx, rank):\n    configure(root, ctx)\n    # Reuse the actual NB5 scorer, stopping before it writes the core verdict/autopsy.\n    path = root / "notebooks/05_evaluate_and_verdict.py"\n    source = path.read_text(encoding="utf-8").split(\n        "scores_ft, preds_ft, rpreds_ft = score_adapter", 1)[0]\n    if "def score_adapter(" not in source:\n        raise RuntimeError("NB5 scorer was not found.")\n    ns = {"__name__": "__bonus_rank_eval__", "__file__": str(path)}\n    exec(compile(source, str(path), "exec"), ns)\n    scores, preds, regression_preds = ns["score_adapter"](\n        root / f"adapters/bonus_rank_{rank}", ns["generate"].NAIVE_PROMPT,\n        label=f"rank-{rank}")\n    ns["report"].write_json(\n        {"rank": rank, "fingerprint": ctx["fingerprint"], "scores": scores.as_dict(),\n         "target_predictions": preds, "regression_predictions": regression_preds},\n        f"bonus_rank_{rank}.json", results_dir=root / "results")\n\n\ndef sweep(root, ctx):\n    script = Path(__file__).resolve()\n    for rank in (8, 64):\n        key = f"bonus_rank_{rank}"\n        adir = root / "adapters" / key\n        if (adir / "adapter_model.safetensors").exists():\n            require_same(read_json(adir / "bonus_context.json"), ctx)\n            print(f"Resume: {key} already trained", flush=True)\n        else:\n            subprocess.run([sys.executable, "-u", str(script), "--train", str(rank)],\n                           cwd=root, check=True)\n        metrics = root / "results" / f"{key}.json"\n        if metrics.exists():\n            require_same(read_json(metrics), ctx)\n        else:\n            subprocess.run([sys.executable, "-u", str(script), "--score", str(rank)],\n                           cwd=root, check=True)\n        require_same(context(root), ctx)\n\n    with (root / "results/bonus_rank_runs.csv").open(encoding="utf-8") as f:\n        trained = {r["run"]: r for r in csv.DictReader(f)}\n    core_scores = read_json(root / "results/verdict.json")["comparison"][2]\n    table = []\n    for rank in (8, 16, 64):\n        run = ctx["core"] if rank == 16 else trained[f"bonus_rank_{rank}"]\n        scores = core_scores if rank == 16 else read_json(\n            root / f"results/bonus_rank_{rank}.json")["scores"]\n        table.append({"rank": rank, "alpha": 2 * rank, "placement": "text-linear",\n                      "lr": float(run["learning_rate"]), "max_steps": int(run["max_steps"]),\n                      "trainable_params": int(run["trainable_params"]),\n                      "train_loss": float(run["final_loss"]),\n                      "peak_vram_gb": float(run["peak_vram_gb"]),\n                      **{k: scores[k] for k in ["target", "regression", "format", "latency_ms", "n"]}})\n    from labkit import report\n    report.write_json({"context": ctx, "rank16_reused_from_core": True, "comparison": table},\n                      "bonus_rank_sweep.json", results_dir=root / "results")\n    print(report.markdown_table(table), flush=True)\n    print("B4 finished: results/bonus_rank_sweep.json", flush=True)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    group = parser.add_mutually_exclusive_group()\n    group.add_argument("--train", type=int, choices=(8, 64))\n    group.add_argument("--score", type=int, choices=(8, 64))\n    args = parser.parse_args()\n    root = Path.cwd()\n    if not (root / "src/labkit").is_dir():\n        raise RuntimeError("Run from the lab repo directory, after completing NB1–NB5.")\n    ctx = context(root)\n    configure(root, ctx)\n    if args.train:\n        train_rank(root, ctx, args.train)\n    elif args.score:\n        score_rank(root, ctx, args.score)\n    else:\n        sweep(root, ctx)\n\n\nif __name__ == "__main__":\n    main()\n', encoding="utf-8")

# B1 needs score comparison and adapter hot-swap, not a 9-GB merged model export.
nb6 = ROOT / 'notebooks/06_merge_and_serve.py'
source = nb6.read_text()
source = source.replace('merged.save_pretrained(out); tok.save_pretrained(out)',
                        'print("Skipping full merged-model export; merge proof is saved below.")')
source = source.replace('del merged; generate.free_memory()',
                        'del model, merged\ngenerate.free_memory()')
nb6.write_text(source, encoding='utf-8')
subprocess.run([sys.executable, 'scripts/build_colab.py'], check=True)
subprocess.run([sys.executable, 'scripts/verify.py', '--smoke'], check=True)


GPU: Tesla T4
Lượt mới: /content/lab21_20261007_221152_dd4c92
Sao lưu Drive: /content/drive/MyDrive/Lab21/20261007_221152_dd4c92


CompletedProcess(args=['/usr/bin/python3', 'scripts/verify.py', '--smoke'], returncode=0)

In [3]:
# @title 3. Core NB1 → NB5 — đánh giá đầy đủ, tự sao lưu sau mỗi bước
os.chdir(ROOT)
os.environ.update(COMPUTE_TIER='T4', BASE_MODEL='unsloth/Qwen3.5-4B',
                  EVAL_LIMIT='0', EPOCHS='2', MASK_MODE='assistant-only', FORCE_RETRAIN='1')
os.environ.pop('ONLY', None)
subprocess.run(
    [sys.executable, '-u', 'scripts/colab_run_persistent.py', '--backup-dir', str(BACKUP),
     'nb1', 'nb2', 'nb3', 'nb4', 'nb5'],
    cwd=ROOT, check=True
)
print('CORE ĐÃ XONG. Kết quả đã có trên Drive:', BACKUP)


CORE ĐÃ XONG. Kết quả đã có trên Drive: /content/drive/MyDrive/Lab21/20261007_221152_dd4c92


In [4]:
# @title 4. Gatekeeper + kết quả — report còn mẫu thì FAIL là bình thường lúc này
os.chdir(ROOT)
subprocess.run([sys.executable, 'scripts/verify.py'], cwd=ROOT, check=False)
print((ROOT / 'results/verdict.json').read_text())
print((ROOT / 'results/autopsy.json').read_text())
print('Mở Google Drive > My Drive > Lab21 >', RUN_ID)


{
  "comparison": [
    {
      "run": "(a) base + naive prompt",
      "target": 0.0,
      "regression": 0.7911,
      "format": 0.0,
      "latency_ms": 3278.8,
      "n": 50
    },
    {
      "run": "(b) base + optimized prompt",
      "target": 0.765,
      "regression": 0.7911,
      "format": 1.0,
      "latency_ms": 1038.8,
      "n": 50
    },
    {
      "run": "(c) LoRA fine-tune",
      "target": 0.97,
      "regression": 0.6778,
      "format": 1.0,
      "latency_ms": 1407.4,
      "n": 50
    }
  ],
  "verdict": {
    "passed": false,
    "reasons": [
      "general capability regressed by 0.113 (tolerance 0.020). See deck §6.3 — add 1-5% replay data."
    ],
    "target_delta": 0.20499999999999996,
    "regression_delta": -0.1133333333333334
  },
  "valid_trace_rate": 0.0
}
[
  {
    "run": "correct",
    "target": 0.97,
    "format": 1.0,
    "latency_ms": 1407.4,
    "n": 50
  },
  {
    "run": "attn_only",
    "target": 0.97,
    "format": 1.0,
    "latency_ms": 917